# London Crime Data Explorer

Explore MPS (Metropolitan Police Service) crime data for London. All datasets load directly from GitHub as Parquet files — no API keys, no authentication.

**Datasets covered:**
- Recorded crime at borough, ward, and LSOA level
- Knife crime monthly trends
- Stop and search records
- Violence against women and girls (VAWG)

Data is published by the MPS via the [London Datastore](https://data.london.gov.uk/) under the Open Government Licence v2.0. These are crimes the MPS *recorded*, not all crimes that occurred.

> **Source:** [fenneh/london-crime-data](https://github.com/fenneh/london-crime-data) — updated monthly.

---
## Setup

Install dependencies (run once):

In [ ]:
!pip install polars matplotlib -q

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO = "https://raw.githubusercontent.com/fenneh/london-crime-data/main/data"

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Ready.")

---
## Recorded Crime: Borough Level

Monthly crime counts broken down by borough, major category, and minor category. The file is in **wide format** — one row per (borough, major category, minor category) combination, with a column per month.

In [ ]:
borough = pl.read_parquet(f"{REPO}/recorded-crime-borough.parquet")

print(f"Shape: {borough.shape}")
print(f"\nColumns: {borough.columns}")
borough.head()

In [ ]:
# Identify the monthly count columns (YYYYMM format)
month_cols = sorted(c for c in borough.columns if str(c).isdigit())
print(f"{len(month_cols)} months: {month_cols[0]} → {month_cols[-1]}")

In [ ]:
# Convert to long format for easier analysis
long = borough.unpivot(
    on=month_cols,
    index=["majortext", "minortext", "boroughname"],
    variable_name="month",
    value_name="count",
).with_columns(
    pl.col("month").str.strptime(pl.Date, "%Y%m")
)

print(f"Long format: {long.shape}")
long.head()

### Crime by major category

Total recorded offences across all boroughs, by major offence category.

In [ ]:
by_category = (
    long.group_by("majortext")
    .agg(pl.col("count").sum())
    .sort("count", descending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(by_category["majortext"].to_list()[::-1],
        by_category["count"].to_list()[::-1],
        color="#1d3557")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K"))
ax.set_title(f"Recorded crime by category ({month_cols[0][:4]}-{month_cols[0][4:]} to {month_cols[-1][:4]}-{month_cols[-1][4:]})",
             fontsize=13)
plt.tight_layout()
plt.show()

### Top boroughs by total offences

In [ ]:
by_borough = (
    long.group_by("boroughname")
    .agg(pl.col("count").sum())
    .sort("count", descending=True)
)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(by_borough["boroughname"].to_list()[::-1],
        by_borough["count"].to_list()[::-1],
        color="#457b9d")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
ax.set_title("Total recorded offences by borough", fontsize=13)
plt.tight_layout()
plt.show()

### Knife crime by borough

Filtering the minor category for knife/blade/weapon offences.

In [ ]:
knife_by_borough = (
    long.filter(pl.col("minortext").str.contains(r"(?i)knife|blade|weapon"))
    .group_by("boroughname")
    .agg(pl.col("count").sum())
    .sort("count", descending=True)
)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(knife_by_borough["boroughname"].to_list()[::-1],
        knife_by_borough["count"].to_list()[::-1],
        color="#e63946")
ax.set_title("Knife / blade / weapon offences by borough", fontsize=13)
plt.tight_layout()
plt.show()

---
## Knife Crime Monthly Trend

Pan-London monthly knife crime figures from the MPS Monthly Crime Dashboard. Covers borough and Safer Neighbourhood Team (SNT) level.

In [ ]:
dashboard = pl.read_parquet(f"{REPO}/monthly-crime-dashboard.parquet")

print(f"Shape: {dashboard.shape}")
print(f"Crime types: {dashboard['crime_type'].unique().to_list()}")
dashboard.head()

In [ ]:
# Borough-level totals only (exclude SNT rows to avoid double-counting)
knife_monthly = (
    dashboard.filter(
        (pl.col("crime_type") == "Knife Crime") &
        (pl.col("area_type") == "Borough")
    )
    .group_by("month_year")
    .agg(pl.col("count_of_incidents").sum().alias("incidents"))
    .sort("month_year")
)

dates = knife_monthly["month_year"].to_list()
values = knife_monthly["incidents"].to_list()

fig, ax = plt.subplots()
ax.plot(dates, values, color="#e63946", linewidth=2)
ax.fill_between(dates, values, alpha=0.07, color="#e63946")
ax.set_title("Knife crime incidents — London (borough total)", fontsize=13)
ax.set_ylabel("Incidents")
# Note: February dips are expected — 28 days vs 30/31 for adjacent months
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nPeak month:   {knife_monthly.sort('incidents', descending=True).row(0)}")
print(f"Lowest month: {knife_monthly.sort('incidents').row(0)}")

### Seasonality: average by month of year

February consistently shows the lowest count — this is a day-count artefact (28 days vs 30/31). Adjusting to a daily rate removes most of the dip.

In [ ]:
import calendar

days_in_month = [calendar.monthrange(d.year, d.month)[1] for d in dates]

seasonal = (
    knife_monthly.with_columns(
        pl.col("month_year").dt.month().alias("month_num")
    )
    .group_by("month_num")
    .agg(pl.col("incidents").mean().alias("avg_incidents"))
    .sort("month_num")
)

month_names = [calendar.month_abbr[m] for m in seasonal["month_num"].to_list()]

fig, ax = plt.subplots()
ax.bar(month_names, seasonal["avg_incidents"].to_list(), color="#e63946", alpha=0.8)
ax.set_title("Average knife crime incidents by month of year", fontsize=13)
ax.set_ylabel("Avg incidents")
plt.tight_layout()
plt.show()

---
## Stop and Search

Individual stop and search records. ~1.35 million records combining multiple rolling snapshots.

**Note on columns:** Records from Mar 2024 onwards use `stopdate` (DD/MM/YYYY string); older records use `date` (YYYY-MM-DD). Officer-defined ethnicity is in `ethnicappearance` for newer records and `ethnic_appearance` for older ones.

In [ ]:
stops = pl.read_parquet(f"{REPO}/stop-search.parquet")

print(f"Shape: {stops.shape}")
print(f"Columns: {stops.columns}")

### Officer-defined ethnicity

These categories are the MPS's own officer-defined appearance codes — not self-defined ethnicity. The 6-category coding (Black, White - North European, Asian, etc.) is recorded by the officer at the time of the stop. Only records where this field is populated are shown (~260K of 1.35M total; older records use a different column name).

In [ ]:
ethnicity = (
    stops.filter(
        pl.col("ethnicappearance").is_not_null() &
        (pl.col("ethnicappearance") != "Unknown")
    )
    .group_by("ethnicappearance")
    .agg(pl.len().alias("stops"))
    .sort("stops", descending=True)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(ethnicity["ethnicappearance"].to_list()[::-1],
        ethnicity["stops"].to_list()[::-1],
        color="#2a9d8f")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
ax.set_title("Stop and search by officer-defined ethnicity", fontsize=13)
ax.set_xlabel("Number of stops")
plt.tight_layout()
plt.show()

### Search power (legislation)

The most common powers used. s1 PACE 1984 requires reasonable suspicion; s60 CJPOA 1994 is suspicion-free (requires prior authorisation for a specific area and time).

In [ ]:
leg_col = next((c for c in stops.columns if "legislat" in c.lower()), None)

if leg_col:
    legislation = (
        stops.filter(pl.col(leg_col).is_not_null())
        .group_by(leg_col)
        .agg(pl.len().alias("stops"))
        .sort("stops", descending=True)
        .head(10)
    )
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(legislation[leg_col].to_list()[::-1],
            legislation["stops"].to_list()[::-1],
            color="#2a9d8f")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
    ax.set_title("Stop and search by legislation (top 10)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Legislation column not found. Available columns:", stops.columns)

### Outcomes

In [ ]:
outcome_col = next((c for c in stops.columns if "outcome" in c.lower()), None)

if outcome_col:
    outcomes = (
        stops.filter(pl.col(outcome_col).is_not_null())
        .group_by(outcome_col)
        .agg(pl.len().alias("stops"))
        .sort("stops", descending=True)
        .head(10)
    )
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(outcomes[outcome_col].to_list()[::-1],
            outcomes["stops"].to_list()[::-1],
            color="#2a9d8f")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
    ax.set_title("Stop and search outcomes (top 10)", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Violence Against Women and Girls (VAWG)

MPS VAWG offence counts from Jan 2014. The MPS definition covers domestic abuse, rape, sexual offences, stalking, harassment, forced marriage, and honour-based abuse.

In [ ]:
vawg = pl.read_parquet(f"{REPO}/vawg-offences.parquet")

print(f"Shape: {vawg.shape}")
print(f"Columns: {vawg.columns}")
vawg.head()

In [ ]:
# Identify offence type and count columns
type_col = next((c for c in vawg.columns if "type" in c.lower() or "offence" in c.lower() or "category" in c.lower()), None)
count_col = next((c for c in vawg.columns if "count" in c.lower() or "total" in c.lower() or "offences" in c.lower()), None)

print(f"Likely type column:  {type_col!r}")
print(f"Likely count column: {count_col!r}")

if type_col and count_col:
    by_type = (
        vawg.group_by(type_col)
        .agg(pl.col(count_col).sum().alias("total"))
        .sort("total", descending=True)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(by_type[type_col].to_list()[::-1],
            by_type["total"].to_list()[::-1],
            color="#e76f51")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
    ax.set_title("VAWG offences by type (Jan 2014–Feb 2026)", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Historical data (back to 2008)

The historical borough file covers Apr 2010–Jul 2023. Combine with the current file for the full series.

In [ ]:
def borough_long(parquet_url: str) -> pl.DataFrame:
    df = pl.read_parquet(parquet_url)
    cols = sorted(c for c in df.columns if str(c).isdigit())
    return df.unpivot(
        on=cols,
        index=["majortext", "minortext", "boroughname"],
        variable_name="month",
        value_name="count",
    ).with_columns(pl.col("month").str.strptime(pl.Date, "%Y%m"))

historical = borough_long(f"{REPO}/recorded-crime-borough-historical.parquet")
current = borough_long(f"{REPO}/recorded-crime-borough.parquet")

full = pl.concat([historical, current]).sort("month")

print(f"Full series: {full['month'].min()} → {full['month'].max()}")
print(f"Total rows: {len(full):,}")

In [ ]:
# Pan-London monthly totals across the full series
monthly_total = (
    full.group_by("month")
    .agg(pl.col("count").sum())
    .sort("month")
)

dates = monthly_total["month"].to_list()
values = monthly_total["count"].to_list()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, values, color="#1d3557", linewidth=1.5)
ax.fill_between(dates, values, alpha=0.07, color="#1d3557")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
ax.set_title("Total recorded crime — London (Apr 2010–Feb 2026)", fontsize=13)
plt.tight_layout()
plt.show()

---
## What next?

**Other datasets in this repo:**
- `recorded-crime-ward.parquet` — ward-level breakdown (Feb 2024–Feb 2026)
- `recorded-crime-lsoa.parquet` — LSOA-level breakdown (~5,000 areas)
- `business-crime-offences.parquet` — 2M+ business crime records
- `homicide.parquet` — homicide accused records from 2003
- `thorough-searches.parquet` — more thorough / intimate part searches (MTIPS)

**For live street-level data** (any UK force, by postcode or coordinate):
- [uk-police-api](https://github.com/fenneh/uk-police-api) — Python client for the data.police.uk REST API

**Terminology reference:**
- [NCRS counting rules](https://www.gov.uk/government/publications/counting-rules-for-recorded-crime) — Home Office definitions for all crime categories
- [PACE Code A](https://www.gov.uk/government/publications/pace-code-a-2023) — stop and search powers